# MUSES + CiteRoots — Tutorial Notebook

End-to-end walkthrough of the MUSES benchmark + CiteRoots labeling layers.

This notebook shows you how to:
1. **Reproduce all 22 paper claims** in 30 seconds with `verify.py`
2. **Inspect the released dataset structure** (instance splits, tier targets, candidate pool)
3. **Examine the CiteRoots labels** (rhetoric paper-level + human-gold audit + endorsement pairs)
4. **Score your own retrieval method** against MUSES tiers + the rhetorical CiteRoots slice

**No setup beyond `pip install` is required** — all parquets are pulled from HuggingFace on demand.

## 0. Install dependencies

In [ ]:
%pip install -q pandas pyarrow huggingface_hub

## 1. Reproduce all 22 paper claims with `verify.py`

This script pulls the required parquets from the two HuggingFace datasets and re-derives every numerical claim in the paper (counts, kappas, hit@100s).

In [ ]:
# If running standalone (no clone), download verify.py:
# !curl -sL https://huggingface.co/datasets/anon-muses-neurips/muses/resolve/main/code/verify.py > verify.py

# Then run it:
!python verify.py

Expected output: `[OK]` on all 22 lines.

## 2. Inspect the released MUSES structure

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

MUSES = "anon-muses-neurips/muses"
CITEROOTS = "anon-muses-neurips/citeroots"

splits = pd.read_parquet(hf_hub_download(MUSES, "instance_splits.parquet", repo_type="dataset"))
pool   = pd.read_parquet(hf_hub_download(MUSES, "candidate_pool.parquet",  repo_type="dataset"))

print(f"Total retrieval instances: {len(splits):,}")
print(f"Split sizes: {splits['split'].value_counts().to_dict()}")
print(f"Candidate pool: {len(pool):,} corpusids")
splits.head()

In [ ]:
# Three familiarity tiers — strict subsets
tiers = {
    name: pd.read_parquet(hf_hub_download(MUSES, f"tier_targets/{name}.parquet", repo_type="dataset"))
    for name in ["citenext", "citenew", "citenew_iso"]
}

for name, df in tiers.items():
    print(f"  {name:<14} {len(df):>11,} positive (focal,target) pairs across {df['focal_corpusid'].nunique():>8,} focal papers")

## 3. Inspect the CiteRoots labeling layers

Two complementary layers, both keyed on `(focal_corpusid, candidate_corpusid)`:
- **CiteRoots-Rhetoric**: passage-level rhetorical-role labels (binary ROOT / non-ROOT) for benchmark-aligned focal→cited edges.
- **CiteRoots-Endorsement**: paper-level author-attested generative-inspiration pairs.

In [ ]:
# Rhetorical layer (paper-level aggregated)
rh = pd.read_parquet(hf_hub_download(CITEROOTS, "rhetoric_labels_paper_level.parquet", repo_type="dataset"))
print(f"Rhetoric pairs: {len(rh):,}")
print(f"ROOT count: {(rh['root_label']==1).sum():,} ({(rh['root_label']==1).mean()*100:.2f}% rate)")
rh.head()

In [ ]:
# Author-endorsed layer
endorse = pd.read_parquet(hf_hub_download(CITEROOTS, "endorsement_pairs.parquet", repo_type="dataset"))
print(f"Release-ready endorsement pairs: {len(endorse):,}")
print(f"Unique focal papers: {endorse['focal_corpusid'].nunique():,}")
print(f"Novelty distribution:")
print(f"  in_reading_shadow=1 (Habitual): {(endorse['is_in_reading_shadow']==1).sum()}")
print(f"  in_reading_shadow=0 (CiteNew):  {(endorse['is_in_reading_shadow']==0).sum()}")
endorse.head()

In [ ]:
# Human-gold audit set — what reproduces κ=0.896
gold = pd.read_parquet(hf_hub_download(CITEROOTS, "human_gold_audit.parquet", repo_type="dataset"))

ROOTS = {"TF", "ME", "GM"}
gold["hr"] = gold["human_label"].apply(lambda x: "ROOT" if x in ROOTS else "WEED")
gold["lr"] = gold["llm_subtype"].apply(lambda x: "ROOT" if x in ROOTS else "WEED")

ct = pd.crosstab(gold['hr'], gold['lr'])
print(f"Audit size: {len(gold)} contexts")
print(f"Binary confusion (rows=human, cols=LLM):")
print(ct)

## 4. Score your own method

Format your predictions as a parquet with three columns: `focal_corpusid`, `candidate_corpusid`, `rank` (rank 0 = top-1; lower is better).

In [ ]:
# Build a tiny mock predictions file (for demonstration only — replace with your method's output)
tg = tiers["citenew"]
test_focals = set(splits[splits["split"]=="test"]["focal_corpusid"].astype("int64"))
tg_test = tg[tg["focal_corpusid"].astype("int64").isin(test_focals)]
sample_focals = tg_test["focal_corpusid"].drop_duplicates().head(100).tolist()

rows = []
for fc in sample_focals:
    pos = tg_test[tg_test["focal_corpusid"]==fc]["target_corpusid"].head(3).tolist()
    for r, c in enumerate(pos):
        rows.append({"focal_corpusid": int(fc), "candidate_corpusid": int(c), "rank": r})
    for r in range(len(pos), 1000):
        rows.append({"focal_corpusid": int(fc), "candidate_corpusid": -r, "rank": r})

pd.DataFrame(rows).to_parquet("my_method.parquet", index=False)
print("Wrote my_method.parquet — replace this with your method's actual output")

In [ ]:
# Score against CiteNew (broad familiarity tier)
# Download eval_test_full.py once if running standalone:
# !curl -sL https://huggingface.co/datasets/anon-muses-neurips/muses/resolve/main/code/eval_test_full.py > eval_test_full.py

!python eval_test_full.py --predictions my_method.parquet --tier citenew

In [ ]:
# Score against the rhetorical CiteRoots slice
# !curl -sL https://huggingface.co/datasets/anon-muses-neurips/muses/resolve/main/code/eval_test_full_citeroots.py > eval_test_full_citeroots.py

!python eval_test_full_citeroots.py --predictions my_method.parquet --slice citeroots_new

## 5. Run the open distilled rhetorical judge

If you want to label new citation contexts (not in the released `rhetoric_labels_paper_level.parquet`), run the released distilled Qwen3-8B + LoRA judge.

It accepts a JSONL of `{context_id, focal_corpusid, candidate_corpusid, context_text, target_marker}` records and emits ROOT/non-ROOT predictions plus calibrated probabilities.

In [ ]:
# Example invocation (requires a GPU, ~5 min for first run to download Qwen3-8B base):
# !pip install -q peft transformers torch
# !python judge_inference.py \
#     --adapter-path anon-muses-neurips/citeroots-rhetoric-judge-qwen3-8b \
#     --input contexts.jsonl \
#     --output predictions.parquet

## What's next

- See the [DATASHEET](https://huggingface.co/datasets/anon-muses-neurips/muses/blob/main/DATASHEET.md) for the full Gebru-style data card.
- See the [Croissant manifest](https://huggingface.co/datasets/anon-muses-neurips/muses/blob/main/croissant.json) for machine-readable schema + RAI metadata.
- See the [paper](https://huggingface.co/datasets/anon-muses-neurips/muses) (anonymized at submission, real link at de-anonymization) for full methodology and findings.

Reproducibility is the goal: every numerical claim in the paper traces to a parquet you can download and a script you can run.